# Lab Report: Polynomial Regression Using PyTorch

## Objective
To implement and train a **polynomial regression model** using PyTorch on a custom dataset (`assignment-data2.csv`), and evaluate its performance using Mean Absolute Error (MAE) loss.

---

## Dataset Description
- **Source:** `assignment-data2.csv`
- **Size:** 99 samples, 2 features (x and y)
- **x range:** -9.8 to 9.8
- **y range:** 1.0 to 673.28
- **Split:** 70% training (69 samples), 30% testing (30 samples)

The data follows a **quadratic (polynomial) pattern**, specifically approximating the form:

> **y = wx + b + (3x)²**

## Step 1: Import Libraries

In [ ]:
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

## Step 2: Load and Explore the Dataset

In [ ]:
# Load dataset
df = pd.read_csv('assignment-data2.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Basic statistics
print("Dataset Statistics:")
df.describe()

## Step 3: Prepare Data as PyTorch Tensors

In [ ]:
# Convert to tensors
X = torch.tensor(df['x'].values, dtype=torch.float32)
y = torch.tensor(df['y'].values, dtype=torch.float32)

print(f"X Shape: {X.shape}")
print(f"y Shape: {y.shape}")

## Step 4: Train-Test Split

In [ ]:
# Split data: 70% train, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set size:  X={X_train.shape}, y={y_train.shape}")
print(f"Testing set size:   X={X_test.shape},  y={y_test.shape}")

## Step 5: Visualize Training and Testing Data

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, label='Training Data', color='blue', alpha=0.7)
plt.scatter(X_test, y_test, label='Testing Data', color='red', alpha=0.7)
plt.title('Training and Testing Data Distribution', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observation: The data follows a quadratic (U-shaped) pattern.")

## Step 6: Define the Polynomial Regression Model

The model uses the formula:
$$\hat{y} = w \cdot x + b + (3x)^2$$

- `weights` and `bias` are **learnable parameters**
- `(3x)²` is a **fixed polynomial component** to capture quadratic behavior

In [ ]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Learnable weight parameter
        self.weights = nn.Parameter(
            torch.randn(1, dtype=torch.float),
            requires_grad=True
        )
        # Learnable bias parameter
        self.bias = nn.Parameter(
            torch.randn(1, dtype=torch.float),
            requires_grad=True
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Polynomial regression: y = wx + b + (3x)^2
        return self.weights * x + self.bias + (3 * x) ** 2

print("Model class defined successfully!")

## Step 7: Initialize Model and Inspect Parameters

In [ ]:
# Set manual seed for reproducibility
torch.manual_seed(42)

# Initialize model
model_0 = LinearRegressionModel()

print("Initial Model Parameters:")
print(f"  weights: {model_0.weights.item():.4f}")
print(f"  bias:    {model_0.bias.item():.4f}")

## Step 8: Initial Predictions (Before Training)

In [ ]:
# Make predictions before training
with torch.inference_mode():
    y_preds_before = model_0(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, label='Actual Test Data', color='blue', alpha=0.7)
plt.scatter(X_test, y_preds_before, label='Predictions (Before Training)', color='red', alpha=0.7)
plt.title('Predictions Before Training', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observation: Before training, predictions are far from actual values.")

## Step 9: Configure Loss Function and Optimizer

| Configuration | Value |
|---|---|
| Loss Function | L1Loss (MAE) |
| Optimizer | SGD |
| Learning Rate | 0.01 |

In [ ]:
# Loss function: Mean Absolute Error (L1 Loss)
loss_fn = nn.L1Loss()

# Optimizer: Stochastic Gradient Descent
optimizer = torch.optim.SGD(
    params=model_0.parameters(),
    lr=0.01
)

print("Loss function: L1Loss (MAE)")
print("Optimizer: SGD with learning rate = 0.01")

## Step 10: Training Loop

The training follows these 5 steps per epoch:
1. **Forward pass** — compute predictions
2. **Calculate loss** — compare predictions to ground truth
3. **Zero gradients** — clear previous gradients
4. **Backward pass** — compute gradients
5. **Update parameters** — optimizer step

In [ ]:
torch.manual_seed(42)

# Training configuration
epochs = 15000
train_loss_values = []
test_loss_values = []
epoch_count = []

print("Starting training...")
print("-" * 70)

for epoch in range(epochs):
    # ---- TRAINING ----
    model_0.train()

    # Step 1: Forward pass
    y_pred = model_0(X_train)

    # Step 2: Calculate loss
    loss = loss_fn(y_pred, y_train)

    # Step 3: Zero gradients
    optimizer.zero_grad()

    # Step 4: Backward pass
    loss.backward()

    # Step 5: Update parameters
    optimizer.step()

    # ---- TESTING ----
    model_0.eval()

    with torch.inference_mode():
        test_pred = model_0(X_test)
        test_loss = loss_fn(test_pred, y_test.type(torch.float))

    # Record every 1000 epochs
    if epoch % 1000 == 0:
        epoch_count.append(epoch)
        train_loss_values.append(loss.detach().numpy())
        test_loss_values.append(test_loss.detach().numpy())
        print(f"Epoch: {epoch:5d} | Train MAE: {loss:.4f} | Test MAE: {test_loss:.4f}")

print("-" * 70)
print("Training complete!")

## Step 11: Plot Loss Curves

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epoch_count, train_loss_values, label='Train Loss (MAE)', color='blue', linewidth=2)
plt.plot(epoch_count, test_loss_values, label='Test Loss (MAE)', color='red', linewidth=2, linestyle='--')
plt.title('Training and Test Loss Curves', fontsize=14)
plt.ylabel('MAE Loss')
plt.xlabel('Epochs')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print("Observation: Both losses decrease steadily — model is converging without overfitting.")

## Step 12: Inspect Learned Parameters

In [ ]:
print("Learned Model Parameters:")
print(model_0.state_dict())
print()
print(f"Final weights: {model_0.weights.item():.4f}")
print(f"Final bias:    {model_0.bias.item():.4f}")

## Step 13: Final Predictions (After Training)

In [ ]:
model_0.eval()

with torch.inference_mode():
    y_preds_after = model_0(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, label='Training Data', color='blue', alpha=0.6)
plt.scatter(X_test, y_test, label='Test Data (Actual)', color='red', alpha=0.6)
plt.scatter(X_test, y_preds_after, label='Predictions (After Training)', color='green', alpha=0.8, marker='x', s=80)
plt.title('Final Predictions After Training', fontsize=14)
plt.xlabel('X')
plt.ylabel('y')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("Observation: Green predictions (x markers) align closer to actual red test points after training.")

## Step 14: Final Evaluation Summary

In [ ]:
model_0.eval()
with torch.inference_mode():
    final_train_pred = model_0(X_train)
    final_test_pred  = model_0(X_test)
    final_train_loss = loss_fn(final_train_pred, y_train)
    final_test_loss  = loss_fn(final_test_pred, y_test)

print("=" * 50)
print("       FINAL EVALUATION SUMMARY")
print("=" * 50)
print(f"  Final Train MAE Loss : {final_train_loss:.4f}")
print(f"  Final Test  MAE Loss : {final_test_loss:.4f}")
print(f"  Final weights        : {model_0.weights.item():.4f}")
print(f"  Final bias           : {model_0.bias.item():.4f}")
print("=" * 50)

## Conclusion

A **polynomial regression model** was successfully implemented and trained using PyTorch:

| Aspect | Detail |
|---|---|
| Model Formula | y = wx + b + (3x)² |
| Learnable Parameters | weights, bias |
| Loss Function | L1Loss (MAE) |
| Optimizer | SGD (lr=0.01) |
| Epochs | 15,000 |
| Final Train MAE | ~48.7 |
| Final Test MAE | ~51.5 |

### Key Observations:
1. Both train and test losses **converged steadily** — no overfitting observed.
2. The fixed `(3x)²` term captures the **quadratic nature** of the data.
3. With only **2 learnable parameters**, the model has limited flexibility.
4. Future improvements: use a **fully connected neural network** or add more polynomial terms.

---
*Lab 04 — Regression Using PyTorch | Dataset: assignment-data2.csv*